# PyTorch Ultra Course — All-in-One Notebook

All chapters concatenated into one notebook (expanded edition).

_Generated: 2026-01-25_

## Contents
Open the Jupyter outline to navigate by headings.

---

# Included chapter: 01_Foundations.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

In [ ]:

def seed_all(seed=1234):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(42)

## 1. Tensor creation, dtype, device

Core properties:
- `shape`, `dtype`, `device`
- `requires_grad`
- contiguous vs non-contiguous layouts

In [ ]:

import torch

a = torch.tensor([[1,2,3],[4,5,6]], dtype=torch.float32, device=device)
b = torch.zeros((2,3), device=device)
c = torch.randn((2,3), device=device)
d = torch.arange(0, 12, device=device).reshape(3,4)

print(a)
print("dtype:", a.dtype, "device:", a.device, "shape:", a.shape)
print("contiguous:", d.is_contiguous())

### Common factories
- `zeros/ones/empty`
- `rand/randn/randint`
- `linspace/logspace`
- `eye/diag`
- numpy interoperability (`from_numpy`, `.numpy()` CPU-only)

In [ ]:

x = torch.linspace(0, 1, 5, device=device)
y = torch.randint(low=0, high=10, size=(3,4), device=device)
I = torch.eye(4, device=device)
print(x)
print("y shape:", y.shape)
print("I shape:", I.shape)

## 2. Strides, views, reshape, permute

PyTorch tensors are strided views over storage.
Key APIs:
- `.stride()`
- `.view()` requires contiguous
- `.reshape()` may copy if needed
- `.transpose()` / `.permute()` usually produce non-contiguous views

In [ ]:

t = torch.arange(12, device=device).reshape(3,4)
tT = t.transpose(0,1)
print("t:", t.shape, t.stride(), "contig:", t.is_contiguous())
print("tT:", tT.shape, tT.stride(), "contig:", tT.is_contiguous())

try:
    tT.view(-1)
except RuntimeError as e:
    print("view failed (expected):", str(e).splitlines()[0])

flat = tT.reshape(-1)
print("reshape ok:", flat.shape)

## 3. Broadcasting and reduction

Broadcast aligns dims from the right; dims must match or be 1.
Reductions: `sum/mean/max/min`, `keepdim=True` to preserve dimensions.

In [ ]:

A = torch.randn(10, 1, device=device)
B = torch.randn(1, 20, device=device)
C = A + B
print("broadcast shape:", C.shape)

u = torch.randn(3,4, device=device)
print("sum over dim=1:", u.sum(dim=1).shape)
print("keepdim:", u.sum(dim=1, keepdim=True).shape)

## 4. Indexing, masks, gather/scatter

- boolean masks for filtering
- integer indexing tensors for advanced selection
- `gather` pulls values from indices
- `scatter_` writes values at indices

In [ ]:

x = torch.arange(0, 24, device=device).reshape(2,3,4)
print("x[0]:\n", x[0])
print("x[:,:,1]:\n", x[:,:,1])

mask = x > 10
print("num>10:", int(mask.sum()))

src = torch.tensor([[10,11,12],[20,21,22]], device=device)
ind = torch.tensor([[2,0],[1,1]], device=device)
g = torch.gather(src, dim=1, index=ind)
print("gather:\n", g)

out = torch.zeros_like(src)
out.scatter_(dim=1, index=ind, src=torch.tensor([[9,9],[7,7]], device=device))
print("scatter:\n", out)

## 5. Autograd core

- set `requires_grad=True`
- `loss.backward()` computes gradients
- grads accumulate; clear them between steps
- use `torch.no_grad()` / `torch.inference_mode()` for inference

In [ ]:

x = torch.tensor(2.0, device=device, requires_grad=True)
y = 3*x**2 + 2*x + 1
y.backward()
print("dy/dx:", x.grad)  # 14

## 6. Numerical stability basics

- use logits-based losses (`CrossEntropyLoss`, `BCEWithLogitsLoss`)
- use `logsumexp` instead of `log(sum(exp(x)))`

In [ ]:

v = torch.randn(4, device=device)
stable = torch.logsumexp(v, dim=0)
naive = torch.log(torch.exp(v).sum())
print("abs diff:", float((stable - naive).abs()))

## 7. Seeding and generators

Determinism can be difficult across devices/backends; seed for debugging.

In [ ]:

import random, numpy as np, torch
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print(torch.rand(3, device=device))

---

# Included chapter: 02_Training_and_Data.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

In [ ]:

def seed_all(seed=1234):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(42)

## 1. Dataset and DataLoader fundamentals

- implement custom Dataset
- DataLoader batching, shuffling, workers
- custom `collate_fn` for variable shapes

In [ ]:

from torch.utils.data import Dataset, DataLoader
import torch

class ToyDataset(Dataset):
    def __init__(self, n=2000, d=5):
        self.X = torch.randn(n, d)
        true_w = torch.randn(d, 1)
        self.y = (self.X @ true_w + 0.1*torch.randn(n,1)).squeeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

ds = ToyDataset()
dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=0)
xb, yb = next(iter(dl))
xb.shape, yb.shape

## 2. Reusable training loop template

Includes:
- train/eval modes
- inference_mode eval
- gradient clearing
- metric hooks

In [ ]:

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class MLPRegressor(nn.Module):
    def __init__(self, d=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): 
        return self.net(x).squeeze(-1)

def train_epoch(model, dl, optimizer, loss_fn, device, max_grad_norm=None):
    model.train()
    total, n = 0.0, 0
    for xb, yb in dl:
        xb = xb.to(device); yb = yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total += loss.item() * xb.size(0); n += xb.size(0)
    return total / n

@torch.inference_mode()
def eval_epoch(model, dl, loss_fn, device):
    model.eval()
    total, n = 0.0, 0
    for xb, yb in dl:
        xb = xb.to(device); yb = yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        total += loss.item() * xb.size(0); n += xb.size(0)
    return total / n

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

for epoch in range(5):
    tr = train_epoch(model, dl, opt, loss_fn, device, max_grad_norm=1.0)
    va = eval_epoch(model, dl, loss_fn, device)
    print(epoch, tr, va)

## 3. Loss functions (must-know)

- regression: MSE, SmoothL1
- classification: CrossEntropy (logits + class indices)
- multi-label: BCEWithLogits (logits + {0,1} targets)

In [ ]:

logits = torch.randn(4, 10, device=device)
labels = torch.randint(0, 10, (4,), device=device)
ce = nn.CrossEntropyLoss()(logits, labels)

logits2 = torch.randn(4, 5, device=device)
targets2 = torch.randint(0, 2, (4,5), device=device).float()
bce = nn.BCEWithLogitsLoss()(logits2, targets2)

float(ce), float(bce)

## 4. Optimizers and parameter groups

AdamW is a modern default; exclude biases and norms from weight decay.

In [ ]:

def param_groups_weight_decay(model, weight_decay=0.01):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(param_groups_weight_decay(model, 0.01), lr=3e-4)
[len(pg["params"]) for pg in opt.param_groups], [pg["weight_decay"] for pg in opt.param_groups]

## 5. Schedulers

Schedulers can be decisive for convergence and final quality.

In [ ]:

from torch.optim.lr_scheduler import StepLR

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3)
sched = StepLR(opt, step_size=2, gamma=0.5)

for epoch in range(5):
    lr = opt.param_groups[0]["lr"]
    loss = train_epoch(model, dl, opt, nn.MSELoss(), device)
    sched.step()
    print("epoch", epoch, "lr", lr, "loss", loss)

## 6. AMP, grad accumulation, and clipping (CUDA)

AMP improves throughput on modern GPUs. Use GradScaler to avoid underflow.

In [ ]:

from torch.cuda.amp import autocast, GradScaler

use_amp = torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)

model = MLPRegressor(d=5).to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

accum_steps = 4
opt.zero_grad(set_to_none=True)

for i, (xb, yb) in enumerate(dl):
    xb = xb.to(device); yb = yb.to(device)
    with autocast(enabled=use_amp):
        pred = model(xb)
        loss = loss_fn(pred, yb) / accum_steps
    scaler.scale(loss).backward()
    if (i+1) % accum_steps == 0:
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
    if i > 10:
        break

print("AMP enabled:", use_amp)

## 7. Checkpointing correctly

Include model + optimizer + scheduler + scaler + metadata.

In [ ]:

import torch

ckpt = {
    "model_state": model.state_dict(),
    "opt_state": opt.state_dict(),
    "scaler_state": scaler.state_dict(),
    "epoch": 0,
}
torch.save(ckpt, "checkpoint_demo.pt")
loaded = torch.load("checkpoint_demo.pt", map_location="cpu")
list(loaded.keys())

---

# Included chapter: 03_nn_Module_DeepDive.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Parameters vs buffers

- parameters: trainable tensors returned by `.parameters()`
- buffers: non-trainable state saved in state_dict (e.g., running stats)

In [ ]:

import torch, torch.nn as nn

class Mod(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(3, 4)
        self.register_buffer("scale", torch.tensor(0.5))
    def forward(self, x):
        return self.lin(x) * self.scale

m = Mod().to(device)
print("state_dict keys:", list(m.state_dict().keys()))
print("num params:", sum(p.numel() for p in m.parameters()))

## 2. Weight initialization

`torch.nn.init` supports:
- Xavier/Glorot (tanh-ish)
- Kaiming/He (ReLU-ish)

In [ ]:

import torch.nn.init as init
lin = nn.Linear(128, 128).to(device)
init.kaiming_normal_(lin.weight, nonlinearity="relu")
init.zeros_(lin.bias)
print(lin.weight.mean().item(), lin.weight.std().item())

## 3. Normalization layers

- BatchNorm: uses batch stats (train) and running stats (eval)
- LayerNorm: stable for variable batch sizes (Transformers)
- GroupNorm: good for small batches in vision

In [ ]:

bn = nn.BatchNorm1d(16).to(device)
ln = nn.LayerNorm(16).to(device)
gn = nn.GroupNorm(4, 16).to(device)

x = torch.randn(8, 16, device=device)
print(bn(x).shape, ln(x).shape, gn(x).shape)

## 4. Dropout

Active only in `train()` mode.

In [ ]:

drop = nn.Dropout(p=0.5).to(device)
x = torch.ones(10, device=device)
drop.train(); y1 = drop(x)
drop.eval();  y2 = drop(x)
print("train mean:", y1.mean().item(), "eval mean:", y2.mean().item())

## 5. Hooks: capture activations and gradients

Forward hooks:
- activation statistics
Backward hooks:
- gradient statistics

In [ ]:

import torch.nn.functional as F
act = {}
def save_act(name):
    def hook(mod, inp, out):
        with torch.no_grad():
            act[name] = {"mean": out.mean().item(), "std": out.std().item()}
    return hook

mlp = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2)).to(device)
h = mlp[0].register_forward_hook(save_act("layer0"))
_ = mlp(torch.randn(64,10, device=device))
h.remove()
act

## 6. Debugging NaNs/Infs

- `set_detect_anomaly(True)` for locating the op
- monitor activation and gradient norms
- `torch.isfinite`, `nan_to_num`

In [ ]:

torch.autograd.set_detect_anomaly(False)

x = torch.tensor([1.0, 0.0], device=device, requires_grad=True)
y = torch.log(x)  # log(0) -> -inf
loss = y.sum()
loss.backward()
print("y:", y)
print("grad:", x.grad)
print("isfinite:", torch.isfinite(y))

---

# Included chapter: 04_Vision_TorchVision.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. TorchVision availability and versions

In [ ]:

try:
    import torchvision
    import torchvision.transforms as T
    from torchvision import datasets, models
    print("torchvision:", torchvision.__version__)
except Exception as e:
    print("torchvision not available:", e)

## 2. Minimal CNN (28x28 grayscale)

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

m = SmallCNN().to(device)
m(torch.randn(8,1,28,28, device=device)).shape

## 3. Transfer learning template (ResNet)

Typical workflow:
- load pretrained backbone
- replace classifier head
- freeze backbone; train head
- unfreeze subset; fine-tune with smaller LR

In [ ]:

transfer_learning_template = r'''
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 5

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.to(device)

for name, p in model.named_parameters():
    if not name.startswith("fc."):
        p.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
'''
print(transfer_learning_template)

## 4. Detection and segmentation patterns (overview)

Detection models require a target dict per image:
- boxes: FloatTensor[N,4]
- labels: Int64Tensor[N]
Segmentation uses masks.

You typically need a custom collate_fn returning lists of images/targets.

---

# Included chapter: 05_Audio_TorchAudio.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. TorchAudio availability

In [ ]:

try:
    import torchaudio
    import torchaudio.transforms as AT
    print("torchaudio:", torchaudio.__version__)
except Exception as e:
    print("torchaudio not available:", e)

## 2. Audio loading pattern

`torchaudio.load` returns:
- waveform: [channels, time]
- sample_rate

In [ ]:

audio_overview = r'''
import torchaudio
waveform, sr = torchaudio.load("file.wav")
if waveform.size(0) > 1:
    waveform = waveform.mean(dim=0, keepdim=True)  # mono
'''
print(audio_overview)

## 3. Spectrograms and mel spectrograms

In [ ]:

spec_template = r'''
import torchaudio.transforms as AT
mel = AT.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=80)
x_mel = mel(waveform)  # [C, n_mels, frames]
'''
print(spec_template)

## 4. Audio classification model pattern (CNN on mel)

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F

class AudioCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)),
        )
        self.fc = nn.Linear(32, num_classes)
    def forward(self, x):
        x = self.net(x).squeeze(-1).squeeze(-1)
        return self.fc(x)

m = AudioCNN().to(device)
m(torch.randn(4,1,80,100, device=device)).shape

---

# Included chapter: 06_NLP_Tokenizers_and_TextPipelines.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Tokenization strategies

- character
- word
- subword (BPE/WordPiece/Unigram)

This chapter includes:
- simple word tokenizer
- vocabulary with specials
- padding + attention masks via collate_fn
- educational BPE training + encoding

In [ ]:

import re
from collections import Counter

def simple_word_tokenize(text: str):
    text = text.lower().strip()
    return re.findall(r"[a-z0-9]+|[^\s\w]", text)

class Vocab:
    def __init__(self, tokens, min_freq=1, specials=("<pad>","<unk>","<bos>","<eos>")):
        counts = Counter(tokens)
        self.itos = list(specials)
        for tok, c in counts.most_common():
            if c >= min_freq and tok not in specials:
                self.itos.append(tok)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    def encode(self, tokens, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(t, self.unk_id) for t in tokens]
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        return [self.itos[i] if 0 <= i < len(self.itos) else "<bad>" for i in ids]

corpus = [
    "Hello world!",
    "Hello PyTorch. PyTorch makes tensors and models.",
    "Tokenization turns text into tokens, then ids."
]
tokens = [t for s in corpus for t in simple_word_tokenize(s)]
vocab = Vocab(tokens)
len(vocab.itos), vocab.encode(simple_word_tokenize("Hello world!"), add_bos=True, add_eos=True)

## 2. Collation: padding and attention masks

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i], self.labels[i]

def collate_pad(batch, vocab: Vocab):
    texts, labels = zip(*batch)
    encoded = [vocab.encode(simple_word_tokenize(t), add_bos=True, add_eos=True) for t in texts]
    lengths = torch.tensor([len(e) for e in encoded], dtype=torch.long)
    max_len = int(lengths.max())
    input_ids = torch.full((len(encoded), max_len), vocab.pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(encoded), max_len), dtype=torch.bool)
    for i, e in enumerate(encoded):
        input_ids[i, :len(e)] = torch.tensor(e, dtype=torch.long)
        attention_mask[i, :len(e)] = True
    labels = torch.tensor(labels, dtype=torch.long)
    return input_ids.to(device), attention_mask.to(device), labels.to(device), lengths.to(device)

texts = ["I like PyTorch.", "PyTorch is fast.", "I like models.", "Tokenizers make ids."]
labels = [1, 1, 0, 0]
tds = TextDataset(texts, labels)
tdl = DataLoader(tds, batch_size=2, shuffle=True, collate_fn=lambda b: collate_pad(b, vocab))
batch = next(iter(tdl))
[tuple(x.shape) for x in batch]

## 3. Char tokenizer

In [ ]:

class CharTokenizer:
    def __init__(self, texts, specials=("<pad>","<unk>","<bos>","<eos>")):
        chars = set()
        for t in texts:
            chars.update(list(t))
        self.itos = list(specials) + sorted(chars)
        self.stoi = {c:i for i,c in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(c, self.unk_id) for c in text]
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            if 0 <= i < len(self.itos):
                out.append(self.itos[i])
        return "".join(out)

ctok = CharTokenizer(corpus)
ctok.encode("Hello", add_bos=True, add_eos=True), len(ctok.itos)

## 4. Educational BPE tokenizer

This is an instructional implementation (not optimized). Production tokenizers use specialized libraries.

In [ ]:

from collections import Counter

def bpe_get_stats(words):
    stats = Counter()
    for w, freq in words.items():
        syms = w.split()
        for i in range(len(syms)-1):
            stats[(syms[i], syms[i+1])] += freq
    return stats

def bpe_merge(pair, words):
    a, b = pair
    pat = re.compile(rf'(?<!\S){re.escape(a)}\s+{re.escape(b)}(?!\S)')
    merged = {}
    for w, freq in words.items():
        merged[pat.sub(a+b, w)] = freq
    return merged

class BPETokenizer:
    def __init__(self, vocab, merges, specials=("<pad>","<unk>","<bos>","<eos>")):
        self.specials = list(specials)
        self.itos = self.specials + sorted(vocab - set(self.specials))
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.merges = merges
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    @staticmethod
    def train(texts, num_merges=100, min_freq=1):
        counts = Counter()
        for text in texts:
            for tok in simple_word_tokenize(text):
                counts[tok] += 1
        words = {}
        for w, f in counts.items():
            if f >= min_freq:
                words[" ".join(list(w)) + " </w>"] = f
        merges = []
        for _ in range(num_merges):
            stats = bpe_get_stats(words)
            if not stats: break
            best = max(stats, key=stats.get)
            merges.append(best)
            words = bpe_merge(best, words)
        vocab = set()
        for w in words.keys():
            vocab.update(w.split())
        vocab.add("</w>")
        return BPETokenizer(vocab=vocab, merges=merges)

    def encode_word(self, word):
        syms = list(word) + ["</w>"]
        for a, b in self.merges:
            i = 0
            new = []
            while i < len(syms):
                if i < len(syms)-1 and syms[i] == a and syms[i+1] == b:
                    new.append(a+b)
                    i += 2
                else:
                    new.append(syms[i]); i += 1
            syms = new
        return syms

    def encode(self, text, add_bos=False, add_eos=False):
        toks = []
        if add_bos: toks.append("<bos>")
        for w in simple_word_tokenize(text):
            toks.extend(self.encode_word(w))
        if add_eos: toks.append("<eos>")
        return [self.stoi.get(t, self.unk_id) for t in toks]

bpe = BPETokenizer.train(corpus, num_merges=50)
len(bpe.itos), bpe.encode("Hello PyTorch!", add_bos=True, add_eos=True)[:30]

---

# Included chapter: 07_Transformers_From_Scratch.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Decoder-only LM (educational)

We implement:
- token embeddings
- sinusoidal positional encoding
- TransformerEncoder blocks with a causal mask
- LM head producing vocab logits

This is an instructional scaffold; production LLMs add: dropout, RMSNorm, rotary embeddings, KV-cache, etc.

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TinyDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=256, max_len=256, pad_id=0):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.tok_emb(input_ids)
        x = self.pos(x)
        T = input_ids.size(1)
        causal = torch.triu(torch.ones(T, T, device=input_ids.device), diagonal=1).bool()
        x = self.tr(x, mask=causal)
        return self.lm_head(x)

## 2. Simple tokenizer + dataset construction (char-level)

Use your own corpora for better results.

In [ ]:

class SimpleCharTok:
    def __init__(self, texts):
        chars = set()
        for t in texts: chars.update(list(t))
        self.itos = ["<pad>","<bos>","<eos>"] + sorted(chars)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]
    def encode(self, text, add_bos=False, add_eos=False):
        ids=[]
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(c, self.pad_id) for c in text]
        if add_eos: ids.append(self.eos_id)
        return ids
    def decode(self, ids):
        out=[]
        for i in ids:
            if 0 <= i < len(self.itos) and self.itos[i] not in ("<pad>","<bos>","<eos>"):
                out.append(self.itos[i])
        return "".join(out)

corpus = [
    "Hello world!",
    "Hello PyTorch. PyTorch makes tensors and models.",
    "Tokenization turns text into tokens, then ids."
]
tok = SimpleCharTok(corpus)

def make_lm_data(texts, tokenizer, block_size=64):
    ids=[]
    for t in texts:
        ids += tokenizer.encode(t, add_bos=True, add_eos=True)
    ids = torch.tensor(ids, dtype=torch.long)
    xs, ys = [], []
    step = block_size
    for i in range(0, len(ids) - block_size - 1, step):
        xs.append(ids[i:i+block_size])
        ys.append(ids[i+1:i+block_size+1])
    return torch.stack(xs), torch.stack(ys)

x, y = make_lm_data(corpus * 200, tok, block_size=64)
dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x,y), batch_size=32, shuffle=True)

lm = TinyDecoderLM(vocab_size=len(tok.itos), pad_id=tok.pad_id, max_len=128).to(device)
opt = torch.optim.AdamW(lm.parameters(), lr=3e-4)

for epoch in range(3):
    lm.train()
    total = 0.0
    for xb, yb in dl:
        xb = xb.to(device); yb = yb.to(device)
        logits = lm(xb)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        total += loss.item()
    print("epoch", epoch, "loss", total/len(dl))

## 3. Generation: temperature + top-k sampling

In [ ]:

@torch.inference_mode()
def sample_next(logits, temperature=1.0, top_k=None):
    logits = logits / max(temperature, 1e-8)
    if top_k is not None:
        v, ix = torch.topk(logits, k=top_k)
        mask = torch.full_like(logits, float("-inf"))
        mask.scatter_(0, ix, v)
        logits = mask
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()

@torch.inference_mode()
def generate(model, tokenizer, prompt, max_new_tokens=120, temperature=1.0, top_k=50):
    model.eval()
    ids = tokenizer.encode(prompt, add_bos=True)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :].squeeze(0)
        next_id = sample_next(logits, temperature=temperature, top_k=top_k)
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)
        if next_id == tokenizer.eos_id:
            break
    return tokenizer.decode(x.squeeze(0).tolist())

print(generate(lm, tok, "Hello", temperature=0.9, top_k=20))

## 4. KV-cache concept (important)

In production LLM inference, you do not recompute attention over the entire context each step.
Instead:
- cache past keys/values per layer
- compute new keys/values for the new token
- attend to cached + new

This reduces generation from O(T^2) to O(T) per token for the attention component.
Implementation depends on the attention module you use.

---

# Included chapter: 08_HuggingFace_Transformers.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. HF quickstart (templates)

These snippets typically require internet to download checkpoints. If offline, point to a local path.

In [ ]:

hf_quickstart = r'''
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

prompt = "Hello, my name is"
inputs = tokenizer(prompt, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.8, top_p=0.95)
print(tokenizer.decode(out[0], skip_special_tokens=True))
'''
print(hf_quickstart)

## 2. Fine-tuning a classifier with Trainer (template)

In [ ]:

hf_trainer_template = r'''
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tok(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized = dataset.map(tok, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label","labels")
tokenized.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
acc = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return acc.compute(predictions=preds, references=p.label_ids)

args = TrainingArguments(
    output_dir="out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()
'''
print(hf_trainer_template[:800] + "\n...\n")

## 3. Chat prompt formatting

Chat models define their own templates (system/user/assistant). Many tokenizers include `.apply_chat_template`.
If unavailable, you must follow the model card prompt rules exactly.

In [ ]:

chat_template = r'''
history = []
def build_prompt(history):
    prompt = ""
    for role, text in history:
        if role == "user":
            prompt += f"User: {text}\n"
        else:
            prompt += f"Assistant: {text}\n"
    prompt += "Assistant: "
    return prompt
'''
print(chat_template)

## 4. Accelerate (concept)

Accelerate manages:
- device placement
- mixed precision
- DDP/FSDP setup
- gradient accumulation

Use it when training medium-to-large models.

---

# Included chapter: 09_Performance_Compile_Export_Profiling.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. torch.compile

`torch.compile` can accelerate training/inference by graph compilation.
Best results: stable shapes, minimal Python branching in hot paths.

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F

class TinyMLP(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d*2), nn.GELU(), nn.Linear(d*2, d))
    def forward(self, x):
        return self.net(x)

m = TinyMLP().to(device).eval()
x = torch.randn(2048, 256, device=device)

if hasattr(torch, "compile"):
    cm = torch.compile(m)
    with torch.inference_mode():
        y1 = m(x)
        y2 = cm(x)
    print("compiled max diff:", (y1-y2).abs().max().item())
else:
    print("torch.compile not available.")

## 2. torch.export

Captures an ExportedProgram graph for deployment/transformations.

In [ ]:

import torch

def try_export():
    if not hasattr(torch, "export"):
        print("torch.export not available.")
        return None
    m = TinyMLP().to(device).eval()
    example = (torch.randn(2,256, device=device),)
    try:
        ep = torch.export.export(m, example)
        print("ExportedProgram ok.")
        return ep
    except Exception as e:
        print("export failed:", type(e).__name__, str(e)[:200], "...")
        return None

ep = try_export()

## 3. Profiling with torch.profiler

In [ ]:

import torch.profiler as profiler

m = TinyMLP().to(device)
inp = torch.randn(4096, 256, device=device)

acts = [profiler.ProfilerActivity.CPU]
if torch.cuda.is_available():
    acts.append(profiler.ProfilerActivity.CUDA)

with profiler.profile(activities=acts, record_shapes=True) as prof:
    for _ in range(50):
        _ = m(inp)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

## 4. Checklist

- inference_mode for eval
- AMP for CUDA
- pinned memory + non_blocking transfers
- dataloader worker tuning
- activation checkpointing for large models

---

# Included chapter: 10_Distributed_DDP_FSDP.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. DDP template (torchrun)

Launch:
```bash
torchrun --nproc_per_node=NUM_GPUS train_ddp.py
```

In [ ]:

ddp_script = r'''
# train_ddp.py
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

def main():
    dist.init_process_group(backend="nccl")  # or "gloo"
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    X = torch.randn(4096, 10)
    y = torch.randint(0, 2, (4096,))
    ds = TensorDataset(X, y)
    sampler = DistributedSampler(ds, shuffle=True)
    dl = DataLoader(ds, batch_size=128, sampler=sampler, num_workers=2, pin_memory=True)

    model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)
    model = DDP(model, device_ids=[local_rank])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(5):
        sampler.set_epoch(epoch)
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''
print(ddp_script[:900] + "\n...\n")

## 2. FSDP sketch

FSDP shards parameters/gradients/optimizer state for very large models.
Checkpointing and wrapping policy matter.

In [ ]:

fsdp_script = r'''
# train_fsdp.py (sketch)
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy

def main():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model = nn.Sequential(nn.Linear(4096, 4096), nn.GELU(), nn.Linear(4096, 4096)).to(device)
    wrap_policy = size_based_auto_wrap_policy(min_num_params=int(1e8))
    model = FSDP(model, auto_wrap_policy=wrap_policy)
    # training loop similar to DDP (optimizer, loss, backward, step)
    # checkpointing requires FSDP state_dict utilities.

if __name__ == "__main__":
    main()
'''
print(fsdp_script[:900] + "\n...\n")

---

# Included chapter: 11_TorchFX_TorchFunc_CustomOps.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Torch FX tracing

In [ ]:

import torch
import torch.nn as nn
from torch.fx import symbolic_trace

class FXModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(4, 8)
        self.l2 = nn.Linear(8, 2)
    def forward(self, x):
        return self.l2(torch.relu(self.l1(x)))

gm = symbolic_trace(FXModel())
print(gm.graph)

## 2. torch.func (grad/vmap concept)

Useful for per-sample gradients, Jacobians, vectorization.

In [ ]:

from torch import func

def f(params, x):
    W, b = params
    return (x @ W + b).sum()

W = torch.randn(5, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
x = torch.randn(10, 5)

g = func.grad(f)((W, b), x)
print("dW:", g[0].shape, "db:", g[1].shape)

## 3. C++ extension template (overview)

In [ ]:

cpp_ext_template = r'''
from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CppExtension

setup(
    name="my_ext",
    ext_modules=[CppExtension("my_ext", ["my_ext.cpp"])],
    cmdclass={"build_ext": BuildExtension},
)
'''
print(cpp_ext_template)

---

# Included chapter: 12_Quantization_and_Deployment.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Dynamic quantization (simple CPU path)

Dynamic quantization is often the easiest win for Linear-heavy models.

In [ ]:

import torch
import torch.nn as nn

try:
    import torch.ao.quantization as tq
    print("torch.ao.quantization available")
except Exception as e:
    tq = None
    print("quantization not available:", e)

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2)).eval()

if tq is not None:
    qdyn = tq.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
    x = torch.randn(4,10)
    y = qdyn(x)
    print("dynamic quant output:", y.shape)

## 2. Serving sketch (FastAPI)

Load once, inference_mode in handlers, consider batching.

In [ ]:

fastapi_template = r'''
import torch
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
model = ...  # load model weights
model.eval()

class Inp(BaseModel):
    x: list[float]

@app.post("/predict")
def predict(inp: Inp):
    with torch.inference_mode():
        x = torch.tensor(inp.x).float().unsqueeze(0)
        logits = model(x)
        probs = torch.softmax(logits, dim=-1).squeeze(0).tolist()
    return {"probs": probs}
'''
print(fastapi_template)

## 3. Export overview

- TorchScript (legacy)
- torch.export (modern capture)
- ONNX (interoperability)

---

# Included chapter: 13_Core_API_Compendium.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## How to use this compendium

This notebook is a **high-density map** of the PyTorch API: modules, what they do, and common usage patterns.
It is not a replacement for reading every docs page verbatim; instead it consolidates the mental model and the
most important entry points.

Major sections:
1. Tensor fundamentals & type promotion
2. Core ops: elementwise, reductions, indexing, broadcasting, out/in-place
3. Linear algebra, FFT, special functions
4. Randomness & generators
5. Serialization, checkpoints, state_dict patterns
6. Devices/backends: CUDA, MPS, CPU, pinned memory, streams
7. Debugging: anomaly detection, profiling, determinism
8. Modeling: nn.Module, containers, init, functional vs module
9. Data: datasets, dataloaders, sampling
10. Training: optimizers, schedulers, AMP, grad accumulation/checkpointing
11. Modern compilation/export stack
12. Distributed primitives (overview)
13. Interop: NumPy, DLPack, XLA notes

The goal is that you can open this notebook as a “PyTorch reference atlas”.

## 1. Type promotion, dtypes, and numerical precision

Things that matter:
- float16 vs bfloat16 vs float32
- accumulation dtypes (e.g., matmul in fp16 may accumulate in fp32 depending on backend)
- integer overflow for int32/int64
- type promotion rules for mixed dtypes

Practical defaults:
- training on CUDA: AMP with fp16 or bf16; master weights usually fp32
- inference: bf16/fp16 if supported and validated; otherwise fp32
- NLP: bf16 often more stable than fp16 on many GPUs

Key APIs:
- `torch.set_default_dtype`
- `.to(dtype=...)`, `.type_as(...)`
- `torch.autocast` (AMP)

In [ ]:

import torch
print("default dtype:", torch.get_default_dtype())
a = torch.tensor([1,2,3])        # int64 by default
b = torch.tensor([1.0,2.0,3.0])  # float32 by default
print(a.dtype, b.dtype)
print((a + b).dtype)  # promotion

## 2. In-place ops vs out-of-place ops (and autograd implications)

In-place ops (ending with `_`) can:
- improve memory usage
- but break autograd if they overwrite values needed for backward

Rule of thumb:
- use in-place ops only when you understand the autograd graph implications
- avoid in-place ops on tensors needed for gradient computation unless necessary

Examples:
- `x.add_(y)` is in-place
- `x = x + y` is out-of-place

In [ ]:

import torch
x = torch.randn(3, requires_grad=True)
y = (x * 2).sum()
# x.add_(1.0)  # would modify x in-place; may or may not error depending on graph needs
y.backward()
print("grad:", x.grad)

## 3. The `out=` parameter pattern

Many ops support `out=` to write into a preallocated tensor. This can reduce allocations in tight loops.

Example: `torch.add(a, b, out=out)`

In [ ]:

import torch
a = torch.randn(5)
b = torch.randn(5)
out = torch.empty(5)
torch.add(a, b, out=out)
out

## 4. Linear algebra (torch.linalg)

Key functions:
- solves: `solve`, `lstsq`
- decompositions: `svd`, `qr`, `cholesky`, `eig`, `eigh`
- matrix functions: `matrix_exp`
- norms: `norm`, `vector_norm`, `matrix_norm`

Use `torch.linalg` for modern, consistent linalg behavior.

In [ ]:

import torch
A = torch.randn(5,5)
A = A @ A.T + 1e-3*torch.eye(5)  # make SPD-ish
L = torch.linalg.cholesky(A)
recon = L @ L.T
print("recon error:", (A-recon).abs().max().item())

x = torch.randn(5)
sol = torch.linalg.solve(A, x)
print("solve residual:", (A@sol - x).norm().item())

## 5. FFT and signal processing

- `torch.fft.fft`, `rfft`, `fft2`, etc.
- useful in audio/vision and some physics-informed models

In [ ]:

import torch
t = torch.linspace(0, 1, 256)
sig = torch.sin(2*math.pi*10*t) + 0.5*torch.sin(2*math.pi*40*t)
spec = torch.fft.rfft(sig)
spec.abs().shape

## 6. Sparse tensors (overview)

PyTorch supports multiple sparse layouts (COO, CSR, CSC, etc.). Use sparse when:
- the tensor is truly sparse
- operations you need are supported for that sparse layout

Caveat: not all ops support sparse; verify compatibility.

In [ ]:

import torch
idx = torch.tensor([[0, 1, 1],
                    [2, 0, 2]])
vals = torch.tensor([3.0, 4.0, 5.0])
S = torch.sparse_coo_tensor(idx, vals, (2,3))
print(S)
print("dense:\n", S.to_dense())

## 7. Randomness: Generator, distributions, and determinism

- `torch.Generator` allows isolated RNG streams
- distributions: `torch.distributions` module
- determinism flags: `torch.use_deterministic_algorithms(True)` (may throw if unsupported)

For debugging: enable determinism; for production training: evaluate tradeoffs.

In [ ]:

import torch
g = torch.Generator().manual_seed(0)
print(torch.rand(3, generator=g))
print(torch.rand(3, generator=g))

## 8. Serialization: tensors, state_dict, and checkpoints

- `torch.save(obj, path)` uses pickle under the hood
- use `state_dict()` for models/optimizers
- prefer `map_location` for portability
- consider `safetensors` for safer, faster model weights in some workflows

In [ ]:

import torch, torch.nn as nn
m = nn.Linear(3,4)
sd = m.state_dict()
torch.save(sd, "linear_sd.pt")
sd2 = torch.load("linear_sd.pt", map_location="cpu")
m2 = nn.Linear(3,4)
m2.load_state_dict(sd2)
print("ok")

## 9. Device management

Common patterns:
- move model and batches to device
- pinned memory in DataLoader (CUDA)
- `non_blocking=True` for faster host→GPU copies when using pinned memory
- manage precision with AMP/autocast

## 10. Modern compilation and export stack (summary)

- `torch.compile` for speed (training/inference)
- `torch.export` for capture and transformations
- `torch.profiler` for profiling

If your goal is deployment, start with `torch.export` and validate correctness.

## 11. Quick index of core modules

- `torch`: tensor + ops + autograd
- `torch.nn`: layers + modules
- `torch.nn.functional`: stateless layer functions
- `torch.optim`: optimizers
- `torch.utils.data`: Dataset/DataLoader
- `torch.distributed`: distributed training
- `torch.cuda`: CUDA-specific utilities
- `torch.backends`: backend configs
- `torch.profiler`: performance profiling
- `torch.ao.quantization`: quantization
- `torch.fx`: graph tracing and transforms
- `torch.func`: vmap/grad/jacobians (advanced)
- `torch.export`: deployment capture

Use this as your mental table of contents.

---

# Included chapter: 14_Advanced_Autograd_and_Checkpointing.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Custom autograd function

Use `torch.autograd.Function` when:
- you need a custom forward that PyTorch cannot differentiate automatically
- you want a custom backward for performance or stability
- you need to wrap external libraries

Rules:
- save tensors for backward using `ctx.save_for_backward`
- backward must return grads for each input

In [ ]:

import torch

class SwishFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sig = torch.sigmoid(x)
        ctx.save_for_backward(sig)
        return x * sig
    @staticmethod
    def backward(ctx, grad_out):
        (sig,) = ctx.saved_tensors
        # d/dx (x*sigmoid(x)) = sig + x*sig*(1-sig)
        # grad = grad_out * (sig + x*sig*(1-sig))
        # we need x, but we didn't save x; reconstruct is not possible => save x too in real impl
        # For demonstration: save x as well
        raise RuntimeError("Demonstration: see improved version below.")

class SwishFn2(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sig = torch.sigmoid(x)
        ctx.save_for_backward(x, sig)
        return x * sig
    @staticmethod
    def backward(ctx, grad_out):
        x, sig = ctx.saved_tensors
        grad = sig + x * sig * (1 - sig)
        return grad_out * grad

def swish(x):
    return SwishFn2.apply(x)

x = torch.randn(5, requires_grad=True)
y = swish(x).sum()
y.backward()
x.grad

## 2. Higher-order gradients

Some applications:
- meta-learning
- hyperparameter optimization
- curvature estimation

Use `create_graph=True` to keep graph for further differentiation.

In [ ]:

import torch
x = torch.randn(3, requires_grad=True)
y = (x**3).sum()
g1 = torch.autograd.grad(y, x, create_graph=True)[0]
g2 = torch.autograd.grad(g1.sum(), x)[0]  # second derivative
g1, g2

## 3. Gradient checkpointing (activation checkpointing)

Checkpointing trades compute for memory by recomputing forward activations in backward.
Useful for large Transformers and deep nets.

API: `torch.utils.checkpoint.checkpoint(function, *args)`

In [ ]:

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

class DeepBlock(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.l1 = nn.Linear(d, d)
        self.l2 = nn.Linear(d, d)
    def forward(self, x):
        return torch.relu(self.l2(torch.relu(self.l1(x))))

block = DeepBlock(128)

def run_with_ckpt(x):
    return checkpoint(block, x)

x = torch.randn(32, 128, requires_grad=True)
y = run_with_ckpt(x).sum()
y.backward()
x.grad.norm()

## 4. Stability toolkit (advanced)

- gradient clipping
- loss scaling (AMP)
- anomaly detection for debugging
- clamp/softplus alternatives for constrained params
- log-space computations for probabilities

---

# Included chapter: 15_Advanced_Transformer_Engineering.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Modern decoder block components (practical)

Common choices in modern decoder-only LMs:
- Pre-norm residual blocks (norm before attention/MLP)
- RMSNorm instead of LayerNorm (often)
- GELU/SwiGLU MLP
- Rotary Positional Embeddings (RoPE)
- Causal self-attention with KV-cache for generation
- Dropout in training; none in inference

This notebook provides **educational implementations** you can adapt.

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F
import math

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d))
    def forward(self, x):
        # x: [..., d]
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.scale

def rotate_half(x):
    x1, x2 = x[..., :x.size(-1)//2], x[..., x.size(-1)//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, sin, cos):
    # q,k: [B, H, T, Dh]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class RoPE(nn.Module):
    def __init__(self, dim, max_len=2048, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)  # [T, dim/2]
        emb = torch.cat([freqs, freqs], dim=-1)       # [T, dim]
        self.register_buffer("sin", emb.sin()[None, None, :, :])  # [1,1,T,dim]
        self.register_buffer("cos", emb.cos()[None, None, :, :])  # [1,1,T,dim]
    def forward(self, T):
        return self.sin[:, :, :T, :], self.cos[:, :, :T, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, rope: RoPE=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.rope = rope

    def forward(self, x, kv_cache=None):
        # x: [B, T, d]
        B, T, d = x.shape
        qkv = self.qkv(x)  # [B, T, 3d]
        q, k, v = qkv.chunk(3, dim=-1)
        # to heads: [B, H, T, Dh]
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1,2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1,2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1,2)

        if self.rope is not None:
            sin, cos = self.rope(T)
            q, k = apply_rope(q, k, sin, cos)

        # KV-cache: append keys/values for generation
        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k], dim=2)
            v = torch.cat([v_prev, v], dim=2)

        # causal attention: compute scores for each head
        # q: [B,H,Tq,Dh], k: [B,H,Tk,Dh]
        Tk = k.size(2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # [B,H,T,Tk]
        causal = torch.triu(torch.ones(T, Tk, device=x.device), diagonal=1 + (Tk - T)).bool()
        att = att.masked_fill(causal, float("-inf"))
        probs = torch.softmax(att, dim=-1)
        y = probs @ v  # [B,H,T,Dh]
        y = y.transpose(1,2).contiguous().view(B, T, d)
        y = self.out(y)

        new_cache = (k, v) if kv_cache is not None else None
        return y, new_cache

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, rope=None):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, rope=rope)
        self.norm2 = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)
    def forward(self, x, kv_cache=None):
        a, new_cache = self.attn(self.norm1(x), kv_cache=kv_cache)
        x = x + a
        m = self.mlp(self.norm2(x))
        x = x + m
        return x, new_cache

## 2. Assemble a decoder-only LM using the advanced block

In [ ]:

class GPTMini(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=4, d_ff=768, max_len=512, pad_id=0):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.rope = RoPE(dim=d_model//n_heads, max_len=max_len)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, rope=self.rope) for _ in range(n_layers)])
        self.norm = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, kv_cache=None):
        # kv_cache: list[(k,v)] per layer for generation; None for training
        x = self.tok(input_ids)
        new_cache = [] if kv_cache is not None else None
        for i, blk in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, c = blk(x, kv_cache=layer_cache)
            if kv_cache is not None:
                new_cache.append(c)
        x = self.norm(x)
        logits = self.head(x)
        return logits, new_cache

# quick shape check
vocab_size = 128
m = GPTMini(vocab_size=vocab_size, max_len=128).to(device)
ids = torch.randint(0, vocab_size, (2, 16), device=device)
logits, _ = m(ids)
logits.shape

## 3. KV-cache generation skeleton

In training, you feed full sequences and do not use KV-cache.
In generation, you typically:
- run full prompt once to initialize cache
- then feed one token at a time, using cache for speed

In [ ]:

@torch.inference_mode()
def gpt_generate(model, prompt_ids, max_new=50, temperature=1.0, top_k=50):
    model.eval()
    x = prompt_ids.unsqueeze(0)  # [1,T]
    # initialize cache with full prompt
    logits, cache = model(x, kv_cache=[(None,None)]*len(model.blocks))  # placeholder, will be handled below

The fully correct KV-cache interface is model-specific and requires careful handling of the initial cache.
Use this notebook as a blueprint for the architecture pieces (RMSNorm, RoPE, SwiGLU, causal attention).
For practical chatbot building, it is standard to rely on Hugging Face generation which already includes KV-cache support.

---

# Included chapter: 16_End_to_End_Project_Recipes.ipynb

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## Projects in this notebook

1. Tabular classifier (MLP) with good training hygiene
2. Vision transfer learning (ResNet) skeleton
3. Audio classifier (mel + CNN) skeleton
4. Text classifier (bag-of-embeddings) in pure PyTorch
5. Mini chatbot (decoder-only LM) training + generation (educational scale)

Each recipe includes:
- model
- data pipeline
- training loop
- evaluation hooks
- checkpointing

## 1. Text classifier: embedding + pooling + MLP (pure PyTorch)

This is a strong baseline and teaches the key parts:
- vocab/tokenizer
- padding + masks
- embedding layers
- pooling strategies (mean/max/attention pooling)

In [ ]:

import torch, torch.nn as nn
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader

def tokenize(text):
    return re.findall(r"[a-z0-9]+|[^\s\w]", text.lower())

class Vocab:
    def __init__(self, texts, min_freq=1, specials=("<pad>","<unk>")):
        counts = Counter()
        for t in texts:
            counts.update(tokenize(t))
        self.itos = list(specials) + [w for w,c in counts.items() if c>=min_freq and w not in specials]
        self.stoi = {w:i for i,w in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
    def encode(self, text):
        return [self.stoi.get(w, self.unk_id) for w in tokenize(text)]

class TxtDS(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts; self.labels = labels; self.vocab = vocab
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        return self.vocab.encode(self.texts[i]), int(self.labels[i])

def collate(batch, pad_id):
    seqs, labels = zip(*batch)
    lens = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    T = int(lens.max())
    x = torch.full((len(seqs), T), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), T), dtype=torch.bool)
    for i,s in enumerate(seqs):
        x[i,:len(s)] = torch.tensor(s, dtype=torch.long)
        mask[i,:len(s)] = True
    return x.to(device), mask.to(device), torch.tensor(labels, dtype=torch.long).to(device)

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, d=128, num_classes=2, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d, padding_idx=pad_id)
        self.mlp = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, num_classes))
    def forward(self, input_ids, mask):
        e = self.emb(input_ids)             # [B,T,d]
        mask_f = mask.unsqueeze(-1).float()
        pooled = (e * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        return self.mlp(pooled)

texts = ["I love PyTorch", "This is bad", "PyTorch is great", "I dislike bugs"]
labels = [1,0,1,0]
vocab = Vocab(texts)
ds = TxtDS(texts, labels, vocab)
dl = DataLoader(ds, batch_size=2, shuffle=True, collate_fn=lambda b: collate(b, vocab.pad_id))

model = TextClassifier(len(vocab.itos), pad_id=vocab.pad_id).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

for epoch in range(50):
    model.train()
    for xb, mb, yb in dl:
        logits = model(xb, mb)
        loss = nn.CrossEntropyLoss()(logits, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

@torch.inference_mode()
def predict(text):
    ids = vocab.encode(text)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    m = torch.ones_like(x, dtype=torch.bool)
    p = torch.softmax(model(x,m), dim=-1)[0]
    return p.tolist()

predict("PyTorch is great")